# Lab 7 — SOLUTION

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import recall_score, precision_score
from fairlearn.metrics import MetricFrame

df = pd.read_csv("data/adult_like.csv")
X = pd.get_dummies(df.drop(columns=["income_gt_50k"]), drop_first=True)
y = df["income_gt_50k"]
Xtr, Xte, ytr, yte, gtr, gte = train_test_split(X, y, df["group"], test_size=0.3, random_state=42)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
pred = model.predict(Xte)

mf = MetricFrame(metrics={"selection_rate": lambda y, p: (p == 1).mean(),
                          "TPR": recall_score, "precision": precision_score},
                y_true=yte, y_pred=pred, sensitive_features=gte)
print(mf.by_group.round(3))
print("gaps:\n", mf.difference().round(3))

**Teaching notes:** the gap table is the artefact. Interpretation practice: 'TPR is X points lower for group B — among truly high earners, B is approved less often' — plain English, numbers attached.